In [13]:
from cobra.io import read_sbml_model
import requests
import os
import pandas as pd

In [14]:
ecoli = read_sbml_model('/Users/edwin/Downloads/iML1515.xml')
ecoli.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.004565,0,0.00%
cl_e,EX_cl_e,0.004565,0,0.00%
cobalt2_e,EX_cobalt2_e,2.192E-05,0,0.00%
cu2_e,EX_cu2_e,0.0006218,0,0.00%
fe2_e,EX_fe2_e,0.01409,0,0.00%
glc__D_e,EX_glc__D_e,10,6,100.00%
k_e,EX_k_e,0.1712,0,0.00%
mg2_e,EX_mg2_e,0.007608,0,0.00%
mn2_e,EX_mn2_e,0.000606,0,0.00%
mobd_e,EX_mobd_e,6.139E-06,0,0.00%


In [15]:
url = "http://bigg.ucsd.edu/api/v2/models"
data = requests.get(url).json()
ecoli_models = [
    m["bigg_id"]
    for m in data["results"]
    if "Escherichia coli" in m["organism"]
]

print(len(ecoli_models))

58


In [16]:
# API endpoint
url = "http://bigg.ucsd.edu/api/v2/models"

# get model list
data = requests.get(url).json()
models = [m["bigg_id"] for m in data["results"]]

# create folder
os.makedirs("bigg_models", exist_ok=True)

# download models
for model in models:
    download_url = f"http://bigg.ucsd.edu/static/models/{model}.xml"
    
    r = requests.get(download_url)
    
    if r.status_code == 200:
        with open(f"bigg_models/{model}.xml", "wb") as f:
            f.write(r.content)
        print(f"Downloaded {model}")
    else:
        print(f"Failed {model}")

Downloaded e_coli_core
Downloaded iAB_RBC_283
Downloaded iAF1260
Downloaded iAF1260b
Downloaded iAF692
Downloaded iAF987
Downloaded iAM_Pb448
Downloaded iAM_Pc455
Downloaded iAM_Pf480
Downloaded iAM_Pk459
Downloaded iAM_Pv461
Downloaded iAPECO1_1312
Downloaded iAT_PLT_636
Downloaded iB21_1397
Downloaded iBWG_1329
Downloaded ic_1306
Downloaded iCHOv1
Downloaded iCHOv1_DG44
Downloaded iCN718
Downloaded iCN900
Downloaded iE2348C_1286
Downloaded iEC042_1314
Downloaded iEC1344_C
Downloaded iEC1349_Crooks
Downloaded iEC1356_Bl21DE3
Downloaded iEC1364_W
Downloaded iEC1368_DH5a
Downloaded iEC1372_W3110
Downloaded iEC55989_1330
Downloaded iECABU_c1320
Downloaded iECB_1328
Downloaded iECBD_1354
Downloaded iECD_1391
Downloaded iECDH10B_1368
Downloaded iEcDH1_1363
Downloaded iECDH1ME8569_1439
Downloaded iEcE24377_1341
Downloaded iECED1_1282
Downloaded iECH74115_1262
Downloaded iEcHS_1320
Downloaded iECIAI1_1343
Downloaded iECIAI39_1322
Downloaded iECNA114_1301
Downloaded iECO103_1326
Downloaded iE

In [17]:
directory_path = '/Users/edwin/Bigg_models/'  
extension = '.xml' 
def xml_dict(directory_path, extension = '.xml'):
    directory_path = directory_path  
    extension = extension 

    # List all files in the directory with the specified extension
    files = [file for file in os.listdir(directory_path) if file.endswith(extension)]

    # get all the files into a dictionary
    microbiome = {}
    
    for xml in files:
        splitnames = xml.split('.') # split file into component
        filename = splitnames[0] 
        microbiome[filename] = directory_path + xml
    
    return microbiome

In [18]:
microbiome	= xml_dict(directory_path, extension)

In [19]:
# =========================================================
# 1. TARGET GENES
# =========================================================
genes = [
    "c-di-gmp", "dgcE", "ydaM",
    "cpsG", "wcaJ",
    "luxS", "lsrA", "lsrB", "lsrC", "lsrD", "lsrK", "lsrR",
    "rpoS",
    "bcsA", "csgA",
    "csrA", "yahA", "gmr", "bcsZ", "pgaB"
]

# Optional: remove c-di-gmp if you only want true genes
# because c-di-gmp is a signaling molecule, not a gene
# genes = [g for g in genes if g != "c-di-gmp"]


# =========================================================
# 2. FUNCTION TO SEARCH EPS/BIOFILM GENES IN ONE MODEL
# =========================================================
def find_matching_genes_in_model(model, target_genes):
    """
    Search for target genes in a COBRA model using both gene.id and gene.name.
    Returns a sorted list of matched target genes.
    """
    found_genes = set()

    for gene_obj in model.genes:
        gene_id = str(gene_obj.id).lower() if gene_obj.id is not None else ""
        gene_name = str(gene_obj.name).lower() if gene_obj.name is not None else ""

        for target in target_genes:
            target_lower = target.lower()

            # Match against gene ID or gene name
            if target_lower in gene_id or target_lower in gene_name:
                found_genes.add(target)

    return sorted(found_genes)


# =========================================================
# 3. LOOP THROUGH E. COLI MODELS
# =========================================================
eps_dict = {}
failed_models = {}

for model_id in ecoli_models:
    try:
        sbml_path = microbiome[model_id]   # assumes microbiome dict maps model_id -> SBML file path
        ecoli_model = read_sbml_model(sbml_path)

        matched = find_matching_genes_in_model(ecoli_model, genes)

        if matched:
            eps_dict[model_id] = matched
        else:
            eps_dict[model_id] = []

    except Exception as e:
        failed_models[model_id] = str(e)


# =========================================================
# 4. COUNT MATCHES PER MODEL
# =========================================================
model_gene_counts = {
    model_id: len(matched_genes)
    for model_id, matched_genes in eps_dict.items()
}

# Sort descending by number of matched genes
sorted_models = sorted(
    model_gene_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

# Top 5 models
top_5_models = sorted_models[:5]


# =========================================================
# 5. PRINT RESULTS
# =========================================================
print("=" * 70)
print("TOP 5 E. coli MODELS WITH MOST EPS/BIOFILM-RELATED GENES")
print("=" * 70)

for rank, (model_id, count) in enumerate(top_5_models, start=1):
    print(f"{rank}. {model_id}")
    print(f"   Matched genes ({count}): {eps_dict[model_id]}")
    print("-" * 70)

print("\nTotal models checked:", len(ecoli_models))
print("Models successfully processed:", len(eps_dict))
print("Models failed:", len(failed_models))

if failed_models:
    print("\nFailed models:")
    for model_id, err in failed_models.items():
        print(f" - {model_id}: {err}")


# =========================================================
# 6. BUILD SUMMARY TABLE
# =========================================================
summary_rows = []

for model_id, matched_genes in eps_dict.items():
    summary_rows.append({
        "model_id": model_id,
        "n_matched_genes": len(matched_genes),
        "matched_genes": ", ".join(matched_genes)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(
    by="n_matched_genes",
    ascending=False
).reset_index(drop=True)

print("\nSummary table (top 10):")
print(summary_df.head(10))


# =========================================================
# 7. SAVE RESULTS
# =========================================================
summary_df.to_csv("ecoli_eps_gene_summary.csv", index=False)

top5_df = summary_df.head(5)
top5_df.to_csv("top5_ecoli_eps_models.csv", index=False)

print("\nSaved:")
print(" - ecoli_eps_gene_summary.csv")
print(" - top5_ecoli_eps_models.csv")

TOP 5 E. coli MODELS WITH MOST EPS/BIOFILM-RELATED GENES
1. iML1515
   Matched genes (11): ['cpsG', 'gmr', 'lsrA', 'lsrB', 'lsrC', 'lsrD', 'lsrK', 'luxS', 'wcaJ', 'yahA', 'ydaM']
----------------------------------------------------------------------
2. iEC55989_1330
   Matched genes (4): ['cpsG', 'lsrC', 'lsrD', 'luxS']
----------------------------------------------------------------------
3. iECDH10B_1368
   Matched genes (4): ['cpsG', 'lsrC', 'lsrD', 'luxS']
----------------------------------------------------------------------
4. iECH74115_1262
   Matched genes (4): ['cpsG', 'lsrC', 'lsrD', 'luxS']
----------------------------------------------------------------------
5. iECIAI1_1343
   Matched genes (4): ['cpsG', 'lsrC', 'lsrD', 'luxS']
----------------------------------------------------------------------

Total models checked: 58
Models successfully processed: 58
Models failed: 0

Summary table (top 10):
         model_id  n_matched_genes  \
0         iML1515               11   


In [20]:
ecoli_model = read_sbml_model('/Users/edwin/bigg_models/iML1515.xml')

In [21]:
gene_map = {}
for gene in ecoli_model.genes:
				gen = ecoli_model.genes.get_by_id(gene.id)
				for eps	in genes:
					if eps in gen.name:
						gene_map[gen.id] = {
							"gene_name": gen.name,
							"reactions": [rxn.id for rxn in gen.reactions],
							"metabolites": [met.id for rxn in gen.reactions for met in rxn.metabolites]
						}

In [22]:
for reaction in ecoli_model.reactions:
				if "biomass" in reaction.name:
					print(reaction.id, reaction.name)

BIOMASS_Ec_iML1515_core_75p37M E. coli biomass objective function (iML1515) - core - with 75.37 GAM estimate
BIOMASS_Ec_iML1515_WT_75p37M E. coli biomass objective function (iML1515) - WT - with 75.37 GAM estimate


In [23]:
from cobra.io import read_sbml_model
from cobra.flux_analysis import pfba
import pandas as pd

model = ecoli_model.copy()

# -------------------------
# 1. Define M9 medium
# -------------------------
def set_m9_glucose_medium(model,
                          glucose_uptake=10.0,
                          oxygen_uptake=20.0,
                          ammonium_uptake=10.0,
                          phosphate_uptake=10.0,
                          sulfate_uptake=10.0):
    # close all exchanges first
    for rxn in model.exchanges:
        rxn.lower_bound = 0.0

    # open standard M9 components
    medium_bounds = {
        "EX_glc__D_e": -glucose_uptake,
        "EX_o2_e": -oxygen_uptake,
        "EX_nh4_e": -ammonium_uptake,
        "EX_pi_e": -phosphate_uptake,
        "EX_so4_e": -sulfate_uptake,
        "EX_h2o_e": -1000.0,
        "EX_h_e": -1000.0,
        "EX_na1_e": -1000.0,
        "EX_k_e": -1000.0,
        "EX_mg2_e": -1000.0,
        "EX_ca2_e": -1000.0,
        "EX_cl_e": -1000.0,
        "EX_fe2_e": -1000.0
    }

    for rxn_id, lb in medium_bounds.items():
        if rxn_id in model.reactions:
            model.reactions.get_by_id(rxn_id).lower_bound = lb

# -------------------------
# 2. Flux readout helper
# -------------------------
tracked_rxns = [
    "BIOMASS_Ec_iML1515_core_75p37M",  # replace if needed
    "PMANM",
    "UDPGPT",
    "DGUNC",
    "CDGUNPD",
    "LDGUNPD",
    "RHCCE",
    "AI2abcpp",
    "AI2K"
]

def collect_fluxes(solution, rxn_ids):
    out = {}
    for rid in rxn_ids:
        out[rid] = solution.fluxes[rid] if rid in solution.fluxes.index else None
    return out

# -------------------------
# 3. EPS optimization
# -------------------------
def optimize_eps_proxy(model, biomass_rxn_id, eps_rxn_id="UDPGPT", frac=0.9):
    # maximize biomass first
    model.objective = biomass_rxn_id
    sol_growth = model.optimize()
    mu_max = sol_growth.objective_value

    # constrain biomass
    biomass_rxn = model.reactions.get_by_id(biomass_rxn_id)
    old_lb = biomass_rxn.lower_bound
    biomass_rxn.lower_bound = frac * mu_max

    # maximize EPS proxy
    model.objective = eps_rxn_id
    sol_eps = model.optimize()

    # reset
    biomass_rxn.lower_bound = old_lb

    return mu_max, sol_growth, sol_eps

# -------------------------
# 4. Run conditions
# -------------------------
results = []

conditions = {
    "baseline_M9": {"glucose": 10, "oxygen": 20, "nh4": 10, "knockdown_cdg_deg": False},
    "high_glucose": {"glucose": 20, "oxygen": 20, "nh4": 10, "knockdown_cdg_deg": False},
    "N_limited": {"glucose": 10, "oxygen": 20, "nh4": 2, "knockdown_cdg_deg": False},
    "high_cdiGMP": {"glucose": 10, "oxygen": 20, "nh4": 10, "knockdown_cdg_deg": True},
    "combined_eps_state": {"glucose": 20, "oxygen": 20, "nh4": 2, "knockdown_cdg_deg": True},
}

biomass_rxn_id = "BIOMASS_Ec_iML1515_core_75p37M"  
for cond_name, pars in conditions.items():
    m = ecoli_model.copy()

    set_m9_glucose_medium(
        m,
        glucose_uptake=pars["glucose"],
        oxygen_uptake=pars["oxygen"],
        ammonium_uptake=pars["nh4"]
    )

    if pars["knockdown_cdg_deg"]:
        for rid in ["CDGUNPD", "LDGUNPD"]:
            if rid in m.reactions:
                rxn = m.reactions.get_by_id(rid)
                rxn.upper_bound = 0.0
                rxn.lower_bound = 0.0

    # pFBA under biomass objective
    m.objective = biomass_rxn_id
    sol = pfba(m)

    row = {"condition": cond_name}
    row.update(collect_fluxes(sol, tracked_rxns))

    # biomass-constrained EPS optimization
    mu_max, sol_growth, sol_eps = optimize_eps_proxy(m, biomass_rxn_id, eps_rxn_id="UDPGPT", frac=0.9)
    row["mu_max"] = mu_max
    row["eps_opt_UDPGPT"] = sol_eps.fluxes["UDPGPT"] if "UDPGPT" in sol_eps.fluxes.index else None
    row["eps_opt_PMANM"] = sol_eps.fluxes["PMANM"] if "PMANM" in sol_eps.fluxes.index else None

    results.append(row)

df = pd.DataFrame(results)
print(df)

            condition  BIOMASS_Ec_iML1515_core_75p37M  PMANM  UDPGPT  DGUNC  \
0         baseline_M9                             0.0    0.0     0.0    0.0   
1        high_glucose                             0.0    0.0     0.0    0.0   
2           N_limited                             0.0    0.0     0.0    0.0   
3         high_cdiGMP                             0.0    0.0     0.0    0.0   
4  combined_eps_state                             0.0    0.0     0.0    0.0   

   CDGUNPD  LDGUNPD         RHCCE  AI2abcpp          AI2K        mu_max  \
0      0.0      0.0  1.391825e-46       0.0  2.445995e-31  0.000000e+00   
1      0.0      0.0  1.391825e-46       0.0  2.445995e-31  0.000000e+00   
2      0.0      0.0  1.391825e-46       0.0  2.445995e-31  0.000000e+00   
3      0.0      0.0  0.000000e+00       0.0  0.000000e+00 -6.858053e-30   
4      0.0      0.0  0.000000e+00       0.0  0.000000e+00 -6.858053e-30   

   eps_opt_UDPGPT  eps_opt_PMANM  
0             0.0            0.0  
1   

In [24]:
"""
eps_experiment_iML1515_gene_map.py

Full EPS-related in silico experiment in an E. coli GSMM
using a gene_map-driven reaction tracking workflow.

What this script does
---------------------
1. Loads an E. coli SBML model (for example iML1515)
2. Uses the model's existing working medium as the baseline
3. Builds a gene_map for EPS/biofilm-related genes:
      gene_id -> gene_name -> reactions -> metabolites
4. Derives tracked reactions directly from gene_map
5. Runs several environmental / regulatory perturbations:
      - baseline
      - high_glucose
      - low_oxygen
      - N_limited
      - high_cdiGMP
      - combined_eps_state
6. For each condition:
      - runs pFBA under biomass objective
      - records fluxes of all tracked reactions
7. Then performs biomass-constrained EPS optimization:
      - maximize biomass
      - constrain biomass to 90% of max
      - maximize UDPGPT (EPS proxy)
8. Saves:
      - full results CSV
      - compact summary CSV
      - gene_map CSV
      - gene_map JSON
      - bar plots

Main biological interpretation
------------------------------
- Strong EPS proxy: UDPGPT (wcaJ-associated)
- Supporting precursor: PMANM (cpsG-associated)
- c-di-GMP regulation: DGUNC, CDGUNPD, LDGUNPD
- AI-2 / quorum sensing branch: RHCCE, AI2abcpp, AI2K

Dependencies
------------
pip install cobra pandas matplotlib

Usage
-----
Edit MODEL_PATH and OUTPUT_DIR, then run:
python eps_experiment_iML1515_gene_map.py
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from cobra.io import read_sbml_model
from cobra.flux_analysis import pfba


# ============================================================
# USER SETTINGS
# ============================================================

MODEL_PATH = "/Users/edwin/bigg_models/iML1515.xml"
OUTPUT_DIR = "/Users/edwin/bigg_models/eps_experiment_output"

# If None, auto-detect from model
BIOMASS_RXN_ID = None

# EPS proxy objective for second-stage optimization
EPS_PROXY_RXN_ID = "UDPGPT"

# Maintain this fraction of max biomass during EPS optimization
BIOMASS_FRACTION_FOR_EPS = 0.90

# Gene keywords / targets used to build gene_map
EPS_GENES = [
    "c-di-gmp", "dgcE", "ydaM",
    "cpsG", "wcaJ",
    "luxS", "lsrA", "lsrB", "lsrC", "lsrD", "lsrK", "lsrR",
    "rpoS",
    "bcsA", "csgA",
    "csrA", "yahA", "gmr", "bcsZ", "pgaB"
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def safe_float(x):
    try:
        if x is None:
            return None
        val = float(x)
        if abs(val) < 1e-12:
            return 0.0
        return val
    except Exception:
        return None


def dedupe_keep_order(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out


# ============================================================
# MODEL / REACTION HELPERS
# ============================================================

def print_candidate_biomass_reactions(model):
    print("\nCandidate biomass reactions:")
    found = False
    for rxn in model.reactions:
        rid = rxn.id.lower()
        rname = (rxn.name or "").lower()
        if "biomass" in rid or "biomass" in rname:
            print(f"  {rxn.id} | {rxn.name}")
            found = True
    if not found:
        print("  None found")


def find_biomass_reaction(model) -> str:
    candidates = []
    for rxn in model.reactions:
        rid = rxn.id.lower()
        rname = (rxn.name or "").lower()
        if "biomass" in rid or "biomass" in rname:
            candidates.append(rxn)

    if not candidates:
        raise ValueError("No biomass reaction found in model.")

    scored = []
    for rxn in candidates:
        score = 0
        rid = rxn.id.lower()
        rname = (rxn.name or "").lower()

        if "iml1515" in rid or "iml1515" in rname:
            score += 5
        if "ec" in rid or "coli" in rid or "escherichia" in rname:
            score += 2
        if "core" in rid or "core" in rname:
            score += 1

        scored.append((score, rxn.id, rxn))

    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return scored[0][2].id


def verify_reactions_exist(model, rxn_ids):
    existing = {r.id for r in model.reactions}
    present = [rid for rid in rxn_ids if rid in existing]
    missing = [rid for rid in rxn_ids if rid not in existing]
    return present, missing


def knockout_reactions(model, rxn_ids):
    for rid in rxn_ids:
        if rid in model.reactions:
            rxn = model.reactions.get_by_id(rid)
            rxn.lower_bound = 0.0
            rxn.upper_bound = 0.0


def get_flux(solution, rxn_id):
    if solution is None:
        return None
    if not hasattr(solution, "fluxes"):
        return None
    if rxn_id not in solution.fluxes.index:
        return None
    return safe_float(solution.fluxes[rxn_id])


def collect_fluxes(solution, rxn_ids):
    out = {}
    for rid in rxn_ids:
        out[rid] = get_flux(solution, rid)
    return out


# ============================================================
# GENE MAP HELPERS
# ============================================================

def classify_gene_role(gene_id: str, gene_name: str) -> str:
    gid = (gene_id or "").lower()
    gname = (gene_name or "").lower()

    token = gid if gid else gname

    if token in {"wcaj", "cpsg", "bcsa", "bcsz", "pgab"} or gname in {"wcaj", "cpsg", "bcsa", "bcsz", "pgab"}:
        return "EPS_structure_or_precursor"

    if token in {"ydam", "yaha", "gmr"} or gname in {"ydam", "yaha", "gmr"}:
        return "c-di-GMP_regulation"

    if token in {"luxs", "lsra", "lsrb", "lsrc", "lsrd", "lsrk", "lsrr"} or gname in {
        "luxs", "lsra", "lsrb", "lsrc", "lsrd", "lsrk", "lsrr"
    }:
        return "quorum_sensing"

    if token in {"rpos", "csra", "csga"} or gname in {"rpos", "csra", "csga"}:
        return "regulatory_or_matrix_associated"

    return "other"


def build_gene_map(model, eps_genes):
    """
    Build:
    gene_map[gene_id] = {
        "gene_name": ...,
        "matched_keyword": ...,
        "role": ...,
        "reactions": [...],
        "metabolites": [...]
    }
    """
    gene_map = {}
    eps_genes_lower = [g.lower() for g in eps_genes]

    for gene in model.genes:
        gen = model.genes.get_by_id(gene.id)
        gene_id = gen.id
        gene_id_lower = gene_id.lower()
        gene_name = gen.name or ""
        gene_name_lower = gene_name.lower()

        matched_keyword = None

        # exact gene id match first
        for eps in eps_genes_lower:
            if gene_id_lower == eps:
                matched_keyword = eps
                break

        # exact gene name match next
        if matched_keyword is None:
            for eps in eps_genes_lower:
                if gene_name_lower == eps:
                    matched_keyword = eps
                    break

        # substring fallback
        if matched_keyword is None:
            for eps in eps_genes_lower:
                if eps and eps in gene_name_lower:
                    matched_keyword = eps
                    break

        if matched_keyword is None:
            continue

        reactions = list(gen.reactions)
        reaction_ids = [rxn.id for rxn in reactions]
        metabolites = sorted({
            met.id
            for rxn in reactions
            for met in rxn.metabolites
        })

        gene_map[gene_id] = {
            "gene_name": gene_name,
            "matched_keyword": matched_keyword,
            "role": classify_gene_role(gene_id_lower, gene_name_lower),
            "reactions": reaction_ids,
            "metabolites": metabolites
        }

    return gene_map


def get_tracked_reactions_from_gene_map(gene_map, biomass_rxn_id=None):
    rxns = []
    if biomass_rxn_id is not None:
        rxns.append(biomass_rxn_id)

    for info in gene_map.values():
        rxns.extend(info["reactions"])

    return dedupe_keep_order(rxns)


def gene_map_to_dataframe(gene_map):
    rows = []
    for gene_id, info in gene_map.items():
        reactions = info.get("reactions", [])
        metabolites = info.get("metabolites", [])

        if reactions:
            for rxn in reactions:
                rows.append({
                    "gene_id": gene_id,
                    "gene_name": info.get("gene_name"),
                    "matched_keyword": info.get("matched_keyword"),
                    "role": info.get("role"),
                    "reaction_id": rxn,
                    "metabolites": ";".join(metabolites)
                })
        else:
            rows.append({
                "gene_id": gene_id,
                "gene_name": info.get("gene_name"),
                "matched_keyword": info.get("matched_keyword"),
                "role": info.get("role"),
                "reaction_id": None,
                "metabolites": ";".join(metabolites)
            })

    return pd.DataFrame(rows)


# ============================================================
# EXPERIMENT HELPERS
# ============================================================

def run_pfba_condition(model, biomass_rxn_id, tracked_rxns):
    model.objective = biomass_rxn_id
    try:
        sol = pfba(model)
    except Exception as e:
        warnings.warn(f"pFBA failed: {e}")
        return None, {
            "status": "pfba_failed",
            "growth": None,
            **{rid: None for rid in tracked_rxns}
        }

    row = {
        "status": getattr(sol, "status", "unknown"),
        "growth": get_flux(sol, biomass_rxn_id)
    }
    row.update(collect_fluxes(sol, tracked_rxns))
    return sol, row


def maximize_eps_given_growth(model, biomass_rxn_id, eps_rxn_id, tracked_rxns, frac=0.90):
    """
    Step 1: maximize biomass
    Step 2: constrain biomass >= frac * mu_max
    Step 3: maximize eps_rxn_id
    """
    m = model.copy()
    m.objective = biomass_rxn_id

    try:
        sol_growth = m.optimize()
    except Exception as e:
        warnings.warn(f"Growth optimization failed: {e}")
        return {
            "mu_max": None,
            "eps_opt_status": "growth_optimization_failed",
            "eps_opt_growth": None,
            "eps_opt_objective": None,
            **{f"eps_opt_{rid}": None for rid in tracked_rxns}
        }

    mu_max = safe_float(sol_growth.objective_value)

    if mu_max is None or mu_max <= 1e-12:
        return {
            "mu_max": 0.0 if mu_max is not None else None,
            "eps_opt_status": "no_growth",
            "eps_opt_growth": 0.0 if mu_max is not None else None,
            "eps_opt_objective": 0.0,
            **{f"eps_opt_{rid}": 0.0 for rid in tracked_rxns}
        }

    if eps_rxn_id not in m.reactions:
        return {
            "mu_max": mu_max,
            "eps_opt_status": f"missing_eps_reaction_{eps_rxn_id}",
            "eps_opt_growth": None,
            "eps_opt_objective": None,
            **{f"eps_opt_{rid}": None for rid in tracked_rxns}
        }

    biomass_rxn = m.reactions.get_by_id(biomass_rxn_id)
    old_lb = biomass_rxn.lower_bound

    try:
        biomass_rxn.lower_bound = frac * mu_max
        m.objective = eps_rxn_id
        sol_eps = m.optimize()

        row = {
            "mu_max": mu_max,
            "eps_opt_status": getattr(sol_eps, "status", "unknown"),
            "eps_opt_growth": get_flux(sol_eps, biomass_rxn_id),
            "eps_opt_objective": safe_float(sol_eps.objective_value)
        }

        for rid in tracked_rxns:
            row[f"eps_opt_{rid}"] = get_flux(sol_eps, rid)

    except Exception as e:
        warnings.warn(f"EPS optimization failed: {e}")
        row = {
            "mu_max": mu_max,
            "eps_opt_status": "eps_optimization_failed",
            "eps_opt_growth": None,
            "eps_opt_objective": None,
            **{f"eps_opt_{rid}": None for rid in tracked_rxns}
        }
    finally:
        biomass_rxn.lower_bound = old_lb

    return row


def build_eps_score(row):
    """
    Heuristic EPS-associated score.
    Weights emphasize EPS initiation first, then precursor/regulatory support.
    """
    udpgpt = safe_float(row.get("eps_opt_UDPGPT")) or 0.0
    pmanm = safe_float(row.get("eps_opt_PMANM")) or 0.0
    dgunc = safe_float(row.get("eps_opt_DGUNC")) or 0.0
    cdgunpd = safe_float(row.get("eps_opt_CDGUNPD")) or 0.0
    ldgunpd = safe_float(row.get("eps_opt_LDGUNPD")) or 0.0
    ai2k = safe_float(row.get("eps_opt_AI2K")) or 0.0
    rhcce = safe_float(row.get("eps_opt_RHCCE")) or 0.0

    score = (
        3.0 * udpgpt +
        1.5 * pmanm +
        1.0 * dgunc +
        0.5 * ai2k +
        0.25 * rhcce -
        1.0 * cdgunpd -
        1.0 * ldgunpd
    )
    return score


def build_dynamic_preferred_cols(df, tracked_rxns):
    cols = [
        "condition",
        "glucose_uptake",
        "oxygen_uptake",
        "ammonium_uptake",
        "phosphate_uptake",
        "sulfate_uptake",
        "knockouts",
        "status",
        "growth",
        "mu_max",
        "eps_opt_status",
        "eps_opt_growth",
        "eps_opt_objective",
        "eps_score",
    ]

    for rid in tracked_rxns:
        if rid in df.columns:
            cols.append(rid)

    for rid in tracked_rxns:
        eps_col = f"eps_opt_{rid}"
        if eps_col in df.columns:
            cols.append(eps_col)

    return dedupe_keep_order(cols)


def save_barplot(df, xcol, ycol, outpath, title=None):
    plot_df = df[[xcol, ycol]].copy().dropna()
    if plot_df.empty:
        return

    plt.figure(figsize=(10, 5))
    plt.bar(plot_df[xcol], plot_df[ycol])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(ycol)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================

def main():
    outdir = ensure_dir(OUTPUT_DIR)

    print(f"Loading model from: {MODEL_PATH}")
    model = read_sbml_model(MODEL_PATH)

    print_candidate_biomass_reactions(model)

    biomass_rxn_id = BIOMASS_RXN_ID or find_biomass_reaction(model)
    print(f"\nUsing biomass reaction: {biomass_rxn_id}")

    base_growth = safe_float(model.slim_optimize())
    print(f"Default model growth before any perturbation: {base_growth}")

    print("\nBaseline model medium:")
    for k, v in sorted(model.medium.items()):
        print(f"  {k}: {v}")

    # --------------------------------------------------------
    # Build gene_map
    # --------------------------------------------------------
    gene_map = build_gene_map(model, EPS_GENES)

    print("\nGene map:")
    if gene_map:
        for gid, info in gene_map.items():
            print(
                f"  {gid} | {info['gene_name']} | "
                f"matched={info['matched_keyword']} | "
                f"role={info['role']} | "
                f"reactions={info['reactions']} | "
                f"metabolites={info['metabolites']}"
            )
    else:
        print("  No matching genes found.")

    # save gene_map as JSON
    gene_map_json_path = outdir / "gene_map.json"
    with open(gene_map_json_path, "w") as f:
        json.dump(gene_map, f, indent=2)
    print(f"\nSaved gene_map JSON to: {gene_map_json_path}")

    # save gene_map as CSV
    gene_map_df = gene_map_to_dataframe(gene_map)
    gene_map_csv_path = outdir / "gene_map.csv"
    gene_map_df.to_csv(gene_map_csv_path, index=False)
    print(f"Saved gene_map CSV to: {gene_map_csv_path}")

    # --------------------------------------------------------
    # Derive tracked reactions from gene_map
    # --------------------------------------------------------
    tracked_rxns = get_tracked_reactions_from_gene_map(
        gene_map,
        biomass_rxn_id=biomass_rxn_id
    )

    print("\nTracked reactions derived from gene_map:")
    for rid in tracked_rxns:
        print(f"  {rid}")

    present, missing = verify_reactions_exist(model, tracked_rxns)
    print("\nTracked reactions present in model:")
    for rid in present:
        print(f"  {rid}")

    if missing:
        print("\nTracked reactions missing from model:")
        for rid in missing:
            print(f"  {rid}")

    # --------------------------------------------------------
    # Experimental conditions
    # --------------------------------------------------------
    base_medium = model.medium.copy()

    conditions = {
        "baseline": {
            "medium_changes": {},
            "knockout_rxns": []
        },
        "high_glucose": {
            "medium_changes": {"EX_glc__D_e": 20.0},
            "knockout_rxns": []
        },
        "low_oxygen": {
            "medium_changes": {"EX_o2_e": 5.0},
            "knockout_rxns": []
        },
        "N_limited": {
            "medium_changes": {"EX_nh4_e": 2.0},
            "knockout_rxns": []
        },
        "high_cdiGMP": {
            "medium_changes": {},
            "knockout_rxns": ["CDGUNPD", "LDGUNPD"]
        },
        "combined_eps_state": {
            "medium_changes": {"EX_glc__D_e": 20.0, "EX_nh4_e": 2.0},
            "knockout_rxns": ["CDGUNPD", "LDGUNPD"]
        }
    }

    rows = []

    for cond_name, cfg in conditions.items():
        print(f"\nRunning condition: {cond_name}")

        m = model.copy()
        medium = base_medium.copy()
        medium.update(cfg["medium_changes"])
        m.medium = medium

        knockout_reactions(m, cfg["knockout_rxns"])

        # stage 1: pFBA at biomass objective
        _, growth_row = run_pfba_condition(
            model=m,
            biomass_rxn_id=biomass_rxn_id,
            tracked_rxns=tracked_rxns
        )

        # stage 2: biomass-constrained EPS optimization
        eps_row = maximize_eps_given_growth(
            model=m,
            biomass_rxn_id=biomass_rxn_id,
            eps_rxn_id=EPS_PROXY_RXN_ID,
            tracked_rxns=tracked_rxns,
            frac=BIOMASS_FRACTION_FOR_EPS
        )

        row = {
            "condition": cond_name,
            "glucose_uptake": medium.get("EX_glc__D_e"),
            "oxygen_uptake": medium.get("EX_o2_e"),
            "ammonium_uptake": medium.get("EX_nh4_e"),
            "phosphate_uptake": medium.get("EX_pi_e"),
            "sulfate_uptake": medium.get("EX_so4_e"),
            "knockouts": ",".join(cfg["knockout_rxns"]) if cfg["knockout_rxns"] else ""
        }

        row.update(growth_row)
        row.update(eps_row)
        row["eps_score"] = build_eps_score(row)
        rows.append(row)

    df = pd.DataFrame(rows)

    preferred_cols = build_dynamic_preferred_cols(df, tracked_rxns)
    final_cols = [c for c in preferred_cols if c in df.columns] + [c for c in df.columns if c not in preferred_cols]
    final_cols = dedupe_keep_order(final_cols)
    df = df[final_cols]

    print("\n================ FINAL RESULTS ================\n")
    print(df.to_string(index=False))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------
    results_csv = outdir / "eps_experiment_results.csv"
    df.to_csv(results_csv, index=False)
    print(f"\nSaved results to: {results_csv}")

    # summary table
    summary_cols = ["condition", "growth", "mu_max", "eps_opt_growth", "eps_opt_objective", "eps_score"]
    for rid in tracked_rxns:
        if rid in df.columns:
            summary_cols.append(rid)
    for rid in tracked_rxns:
        eps_col = f"eps_opt_{rid}"
        if eps_col in df.columns:
            summary_cols.append(eps_col)

    summary_cols = dedupe_keep_order([c for c in summary_cols if c in df.columns])
    summary_df = df[summary_cols].copy()

    summary_csv = outdir / "eps_experiment_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"Saved summary to: {summary_csv}")

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------
    save_barplot(
        df,
        xcol="condition",
        ycol="growth",
        outpath=outdir / "plot_growth.png",
        title="Growth by condition"
    )

    if "eps_opt_UDPGPT" in df.columns:
        save_barplot(
            df,
            xcol="condition",
            ycol="eps_opt_UDPGPT",
            outpath=outdir / "plot_eps_opt_UDPGPT.png",
            title="EPS proxy optimization: UDPGPT"
        )

    if "eps_opt_PMANM" in df.columns:
        save_barplot(
            df,
            xcol="condition",
            ycol="eps_opt_PMANM",
            outpath=outdir / "plot_eps_opt_PMANM.png",
            title="EPS precursor optimization: PMANM"
        )

    if "eps_opt_DGUNC" in df.columns:
        save_barplot(
            df,
            xcol="condition",
            ycol="eps_opt_DGUNC",
            outpath=outdir / "plot_eps_opt_DGUNC.png",
            title="c-di-GMP synthesis flux: DGUNC"
        )

    if "eps_score" in df.columns:
        save_barplot(
            df,
            xcol="condition",
            ycol="eps_score",
            outpath=outdir / "plot_eps_score.png",
            title="EPS-associated score by condition"
        )

    print("\nDone.")


if __name__ == "__main__":
    main()

Loading model from: /Users/edwin/bigg_models/iML1515.xml

Candidate biomass reactions:
  BIOMASS_Ec_iML1515_core_75p37M | E. coli biomass objective function (iML1515) - core - with 75.37 GAM estimate
  BIOMASS_Ec_iML1515_WT_75p37M | E. coli biomass objective function (iML1515) - WT - with 75.37 GAM estimate

Using biomass reaction: BIOMASS_Ec_iML1515_core_75p37M
Default model growth before any perturbation: 0.87699721425716

Baseline model medium:
  EX_ca2_e: 1000.0
  EX_cl_e: 1000.0
  EX_co2_e: 1000.0
  EX_cobalt2_e: 1000.0
  EX_cu2_e: 1000.0
  EX_fe2_e: 1000.0
  EX_fe3_e: 1000.0
  EX_glc__D_e: 10.0
  EX_h2o_e: 1000.0
  EX_h_e: 1000.0
  EX_k_e: 1000.0
  EX_mg2_e: 1000.0
  EX_mn2_e: 1000.0
  EX_mobd_e: 1000.0
  EX_na1_e: 1000.0
  EX_nh4_e: 1000.0
  EX_ni2_e: 1000.0
  EX_o2_e: 1000.0
  EX_pi_e: 1000.0
  EX_sel_e: 1000.0
  EX_slnt_e: 1000.0
  EX_so4_e: 1000.0
  EX_tungs_e: 1000.0
  EX_zn2_e: 1000.0

Gene map:
  b2687 | luxS | matched=luxs | role=quorum_sensing | reactions=['RHCCE'] | met

In [25]:
model.metabolites.udcpp_c.summary(fva=0.95)

/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas require

Percent,Flux,Range,Reaction,Definition
100.00%,0.02437,[0.02315; 1.037],UDCPDP,h2o_c + udcpdp_c --> h_c + pi_c + udcpp_c
0.00%,0,[0; 0.03669],UDCPPtppi,udcpp_p --> udcpp_c
Percent,Flux,Range,Reaction,Definition
0.00%,0,[-0.03669; 0],ACGAMT,uacgam_c + udcpp_c --> ump_c + unaga_c
100.00%,-0.02437,[-1.037; -0.02315],PAPPT3,udcpp_c + ugmda_c --> uagmda_c + ump_c


In [26]:
from cobra.util.solver import linear_reaction_coefficients

model.objective = {
    model.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M"): 1.0,
    model.reactions.get_by_id("UDPGPT"): 0.1
}

In [27]:
print(model.objective)

Maximize
1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685 + 0.1*UDPGPT - 0.1*UDPGPT_reverse_73db4


In [28]:
solution = model.optimize()

print("Objective value:", solution.objective_value)
print("Flux through UDPGPT:", solution.fluxes["UDPGPT"])

Objective value: 0.8769972144269801
Flux through UDPGPT: 0.0


In [29]:
model.objective = "UDPGPT"
sol = model.optimize()

print("Max UDPGPT:", sol.objective_value)
print("UDPGPT flux:", sol.fluxes["UDPGPT"])
print("Biomass flux:", sol.fluxes["BIOMASS_Ec_iML1515_core_75p37M"])

Max UDPGPT: 0.0
UDPGPT flux: 0.0
Biomass flux: 0.8769972144269801


In [30]:
from cobra.flux_analysis import flux_variability_analysis

fva = flux_variability_analysis(model, reaction_list=["UDPGPT"])
print(fva)

        minimum  maximum
UDPGPT      0.0      0.0


In [31]:
for met_id in ["udcpp_c", "udpg_c", "udcpgl_c", "ump_c"]:
    met = model.metabolites.get_by_id(met_id)
    print(f"\nMetabolite: {met.id} | {met.name}")
    print("Reactions:")
    for r in met.reactions:
        print(" ", r.id, ":", r.reaction)


Metabolite: udcpp_c | Undecaprenyl phosphate
Reactions:
  UDCPPtppi : udcpp_p --> udcpp_c
  UPLA4FNT : udcpp_c + udpLa4fn_c --> uLa4fn_c + udp_c
  PAPPT3 : udcpp_c + ugmda_c --> uagmda_c + ump_c
  UDPGPT : udcpp_c + udpg_c <=> udcpgl_c + ump_c
  ACGAMT : uacgam_c + udcpp_c --> ump_c + unaga_c
  UDCPDP : h2o_c + udcpdp_c --> h_c + pi_c + udcpp_c

Metabolite: udpg_c | UDPglucose
Reactions:
  GALT1 : gicolipa_c + udpg_c --> gagicolipa_c + h_c + udp_c
  GALUi : g1p_c + h_c + utp_c --> ppi_c + udpg_c
  UDPGPT : udcpp_c + udpg_c <=> udcpgl_c + ump_c
  UGLT : gal1p_c + udpg_c <=> g1p_c + udpgal_c
  UDPG4E : udpg_c <=> udpgal_c
  TRE6PS : g6p_c + udpg_c --> h_c + tre6p_c + udp_c
  GLCTR1 : icolipa_c + udpg_c --> gicolipa_c + h_c + udp_c
  GLCTR3 : ggagicolipa_c + udpg_c --> gggagicolipa_c + h_c + udp_c
  O16GLCT1 : aragund_c + udpg_c --> garagund_c + h_c + udp_c
  O16GLCT2 : gfgaragund_c + udpg_c --> h_c + o16aund_c + udp_c
  GLCTR2 : gagicolipa_c + udpg_c --> ggagicolipa_c + h_c + udp_c
  UD

In [33]:
from cobra import Reaction

def add_udcpgl_sink(model):
    if "SK_udcpgl_c" not in model.reactions:
        sink = Reaction("SK_udcpgl_c")
        sink.name = "Sink for udcpgl_c"
        sink.lower_bound = 0.0
        sink.upper_bound = 1000.0
        met = model.metabolites.get_by_id("udcpgl_c")
        sink.add_metabolites({met: -1.0})
        model.add_reactions([sink])


m = model.copy()
add_udcpgl_sink(m)

biomass_rxn = "BIOMASS_Ec_iML1515_core_75p37M"
eps_rxn = "UDPGPT"

# Step 1: maximize biomass
m.objective = biomass_rxn
sol_growth = m.optimize()
mu_max = sol_growth.objective_value

print("Max biomass:", mu_max)

# Step 2: constrain biomass
m.reactions.get_by_id(biomass_rxn).lower_bound = 0.9 * mu_max

# Step 3: maximize EPS
m.objective = eps_rxn
sol_eps = m.optimize()

print("EPS objective:", sol_eps.objective_value)
print("Biomass flux:", sol_eps.fluxes[biomass_rxn])
print("UDPGPT flux:", sol_eps.fluxes[eps_rxn])
print("Sink flux:", sol_eps.fluxes["SK_udcpgl_c"])

Max biomass: 0.8769972144270056
EPS objective: 0.06383025325117728
Biomass flux: 0.7892974929843051
UDPGPT flux: 0.06383025325117728
Sink flux: 0.06383025325117728
